This notebook calculates the first year annual and lifetime solar savings by state, for use in the Big Numbers database. 

In [ ]:
import sys, os
import pandas as pd
import importlib

sys.path.insert(0, '../')
import analysis_functions as af
importlib.reload(af)

# --- Configuration ---
DRIVE_ROOT    = "/Users/wael/Library/CloudStorage/GoogleDrive-wael@permitpower.org/Shared drives/PP (All)/Research/$1 watt solar/Results"
ROOT_DIR      = os.path.join(DRIVE_ROOT, "Raw")
RUN_ID        = "run_all_states_net_savings_add_itc"

COHORT_YEAR   = 2026
LIFETIME_YEAR = 2026

STATES = [
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA",
    "HI", "ID", "IL", "IN", "IA", "KS", "KY", "LA", "ME", "MD",
    "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ",
    "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC",
    "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY",
]

cfg = af.SavingsConfig(lifetime_years=25, cap_to_horizon=False)

In [ ]:
# Main output: 2026 adoption cohort anchored stats, baseline scenario only
results = []

for state in STATES:
    print(f"Processing {state}...", end=" ")
    try:
        wh = af.DataWarehouse.from_disk(ROOT_DIR, run_id=RUN_ID, states=[state])

        if wh.agents.empty:
            print("no data found, skipping.")
            results.append({"state_abbr": state.upper(),
                            "median_yr1_bill_savings_2026_cohort": None,
                            "median_lifetime_savings_2026_cohort": None})
            continue

        outputs = af.aggregate_state_metrics(wh.agents, cfg)

        # median_lifetime_savings_per_adopter_to_date for LIFETIME_YEAR, baseline
        df_pas = outputs["portfolio_annual_savings"]
        mask = (df_pas["scenario"] == "baseline") & (df_pas["year"] == LIFETIME_YEAR)
        lifetime_val = df_pas.loc[mask, "median_lifetime_savings_per_adopter_to_date"].iloc[0] \
                       if mask.any() else None

        # median_yr1_bill_savings for baseline (2026 cohort only)
        df_yr1 = outputs["median_yr1_bill_savings_2026_cohort"]
        mask_yr1 = df_yr1["scenario"] == "baseline"
        yr1_val = df_yr1.loc[mask_yr1, "median_yr1_bill_savings_2026_cohort"].iloc[0] \
                  if mask_yr1.any() else None

        results.append({
            "state_abbr":                          state.upper(),
            "median_yr1_bill_savings_2026_cohort":  yr1_val,
            "median_lifetime_savings_2026_cohort":  lifetime_val,
        })
        print(f"done. yr1=${yr1_val:,.0f}  lifetime=${lifetime_val:,.0f}")

    except Exception as e:
        print(f"ERROR: {e}")
        results.append({"state_abbr": state.upper(),
                        "median_yr1_bill_savings_2026_cohort": None,
                        "median_lifetime_savings_2026_cohort": None})

df_results = pd.DataFrame(results)
df_results


In [ ]:
out_path = os.path.join(ROOT_DIR, "state_savings_summary.csv")
df_results.to_csv(out_path, index=False)
print(f"Saved to {out_path}")


## QA/QC: Comparison against reference dataset

The cells below compare the methodology used here against a pre-existing reference dataset
(referred to as the "big numbers database"). That dataset contained fields:
`weighted_avg_bill_without_pv_year1`, `weighted_avg_bill_with_pv_year1`,
`weighted_avg_savings_year1`, `pct_savings_year1`, and `lifetime_savings_weighted`.

**Finding 1 — Scenario:** The reference dataset uses the **policy scenario**, not baseline.

**Finding 2 — Cohort blending:** The reference dataset uses a **blend of all adoption cohorts (2026–2040)**
rather than anchoring to the 2026 cohort. This means later cohorts with higher electricity prices
(due to 3% annual rate escalation + 2.5% inflation) pull up the weighted averages. Specifically:

- `bill_wo` all cohorts (policy): matches reference closely
- `lifetime_savings` all cohorts (policy): ~1.7% off reference — consistent with different model
  parameters in the original run, not a methodology difference

The outputs in this notebook use the **2026 cohort only, baseline scenario**, which is a cleaner
and more defensible approach since it is not influenced by future electricity price levels.

**Finding 3 — Lifetime savings definition:** Confirmed to be gross energy value (`cf_energy_value`,
summed over 25 years), not net of debt payments. Net-of-debt lifetime savings was far from the reference.


In [ ]:
# QA/QC: run for a single state and compare against reference numbers
import ast

CHECK_STATE    = "NC"       # state with known reference values
CHECK_SCENARIO = "baseline" # toggle to "policy" to check

wh_check = af.DataWarehouse.from_disk(ROOT_DIR, run_id=RUN_ID, states=[CHECK_STATE])
agents = wh_check.agents[wh_check.agents["scenario"] == CHECK_SCENARIO].copy()

def parse_arr(v):
    if isinstance(v, list): return v
    try:
        s = str(v).strip()
        if s.startswith("{"): return [float(x) for x in s.strip("{}").split(",") if x.strip()]
        return list(ast.literal_eval(s))
    except: return []

def compute_stats(df, label):
    df = df.copy()
    df["bill_wo_yr1"]    = df["utility_bill_wo_sys_pv_only"].apply(lambda v: parse_arr(v)[1] if len(parse_arr(v)) > 1 else 0.0)
    df["bill_w_yr1"]     = df["utility_bill_w_sys_pv_only"].apply(lambda v: parse_arr(v)[1] if len(parse_arr(v)) > 1 else 0.0)
    df["lifetime_gross"] = df["cf_energy_value_pv_only"].apply(lambda v: sum(parse_arr(v)[:25]))
    w = df["new_adopters"]; total = w.sum()
    wo  = (df["bill_wo_yr1"]    * w).sum() / total
    w_  = (df["bill_w_yr1"]     * w).sum() / total
    sav = wo - w_
    lt  = (df["lifetime_gross"] * w).sum() / total
    print(f"{label:<20} bill_wo=${wo:,.0f}  bill_w=${w_:,.0f}  savings=${sav:,.0f}  pct={sav/wo:.1%}  lifetime=${lt:,.0f}")

print(f"Scenario: {CHECK_SCENARIO}")
compute_stats(agents, "All cohorts:")
compute_stats(agents[agents["year"] == 2026], "2026 cohort only:")
print(f"{'Reference (NC):':<20} bill_wo=$1,529  bill_w=$246  savings=$1,283  pct=83.9%  lifetime=$59,658")
